# METROPULSE — 01. Data Understanding & Ingestion Audit
## DataNova Global Data-thon 2026 | Problem Statement DN-DS-01
This notebook performs an exhaustive technical audit of the **15 raw TfL NUMBAT workbooks** covering 2023, 2024, and 2025.
It covers:
1. Dynamic file discovery and metadata extraction.
2. Multi-year worksheet inventory and schema comparison.
3. Transport mode identification and London Underground (LU) filtering rules.
4. Missing value and data quality assessment.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config.settings import DATA_RAW, DATA_CSV, REPORTS_DIR
from src.ingestion.workbook_reader import discover_files, list_sheets, read_sheet
from src.ingestion.inventory import build_inventory, print_summary

### 1. Discover all Raw Workbooks

In [ ]:
files = discover_files(DATA_RAW)
print(f"Total NUMBAT workbooks discovered: {len(files)}")
pd.DataFrame(files)[["filename", "year", "day_type_code", "day_type_label", "file_size_mb"]]

### 2. Inspect Worksheet Structure across Years

In [ ]:
# Inspect sample sheets across 2023, 2024, 2025
inventory = pd.read_csv(REPORTS_DIR / "data_quality" / "source_inventory.csv")
inventory.groupby("sheet_name")[["present", "row_count", "missing_pct"]].agg({
    "present": "sum",
    "row_count": "mean",
    "missing_pct": "mean"
}).round(2)

### 3. Transport Mode Identification & London Underground (LU) Filtering

In [ ]:
# Demonstrate Mode breakdown in Line_Boarders
sample_file = files[0]["path"]
df_modes = pd.read_excel(sample_file, sheet_name="Line_Boarders", header=2)
print("Modes present in NUMBAT:")
print(df_modes["Mode"].value_counts(dropna=False))
print("\nRetained London Underground (LU) lines:")
print(df_modes[df_modes["Mode"] == "LU"][["Line Code", "Line Name", "Total"]])